# Autoencoder baseline
Runtime > Run all. Allow Google Drive access when asked. It trains, for every dataset and seed, the RF/XGBoost detector and an autoencoder on clean flows, and compares them under the same splits, threshold and metrics. Results are saved to the revision_results folder.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
except ImportError:
    drive = None                      # not running in Colab
if drive is not None:
    drive.mount("/content/drive")     # errors are shown, not hidden
    assert os.path.exists("/content/drive/MyDrive"), "Google Drive did not mount - re-run and click Allow"

# ---------------- CONFIG (edit here) ----------------
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
FAMILIES = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "CICIDS2018": ["Zeus_Ares"]}
ATTACKS = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]
SEEDS = list(range(int(os.environ.get("N_SEEDS", "5"))))
N_CHUNKS = int(os.environ.get("N_CHUNKS", "20"))       # position blocks
BUFFER = 0.02                                          # drop rows within 2% of a chunk edge
MAX_BENIGN = int(os.environ.get("MAX_BENIGN", "150000"))        # systematic (ordered) subsample of benign
MAX_TRAIN_PER_CLASS = int(os.environ.get("MAX_TRAIN", "40000"))
MAX_EVAL_PER_SET = int(os.environ.get("MAX_EVAL", "30000"))     # cap per val/test set (speed)
FAR_TARGET_PCT = 95                                    # threshold = 95th percentile of clean validation scores
BITS = 8                                               # quantization bits for the cleaned-up squeezer
DROP = {"Unnamed: 0", "Label", "label", "n_actions"}
os.makedirs(OUT, exist_ok=True)


In [ ]:
# ---------------- DATA ----------------
def with_pos(df):
    df = df.copy()
    df["_pos"] = (np.arange(len(df)) + 0.5) / len(df)
    return df

def load_dataset(ds):
    d = f"{BASE}/{ds}"
    benign = pd.read_csv(f"{d}/benign.csv")
    if len(benign) > MAX_BENIGN:                       # ordered subsample, keeps temporal structure
        benign = benign.iloc[np.linspace(0, len(benign) - 1, MAX_BENIGN).astype(int)]
    fams = FAMILIES[ds]
    orig = [with_pos(pd.read_csv(f"{d}/malicious/{f}.csv")) for f in fams]
    adv = {(o, a): [with_pos(pd.read_csv(f"{d}/{o}/{a}/{f}.csv")) for f in fams] for o, a in ATTACKS}
    benign = with_pos(benign)
    ref = [benign] + orig + [x for v in adv.values() for x in v]
    feats = [c for c in benign.columns
             if c not in DROP and c != "_pos"
             and all(c in r.columns for r in ref)
             and all(pd.api.types.is_numeric_dtype(r[c]) for r in ref)]
    def pool(frames):
        df = pd.concat(frames, ignore_index=True)
        X = df[feats].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).values.astype(np.float32)
        return X, df["_pos"].values
    data = {"benign": pool([benign]), "orig": pool(orig)}
    for k, v in adv.items():
        data[k] = pool(v)
    return data, feats

def split_roles(seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N_CHUNKS)
    n_tr, n_va = int(N_CHUNKS * 0.5), int(N_CHUNKS * 0.2)
    role = np.empty(N_CHUNKS, dtype=object)
    role[perm[:n_tr]] = "train"
    role[perm[n_tr:n_tr + n_va]] = "val"
    role[perm[n_tr + n_va:]] = "test"
    return role

def take(pool, role, split, rng, cap):
    X, pos = pool
    ch = np.minimum((pos * N_CHUNKS).astype(int), N_CHUNKS - 1)
    frac = pos * N_CHUNKS - ch
    m = (role[ch] == split) & (frac > BUFFER) & (frac < 1 - BUFFER)
    Xs = X[m]
    if len(Xs) > cap:
        Xs = Xs[rng.choice(len(Xs), cap, replace=False)]
    return Xs


In [ ]:
# ---------------- SQUEEZERS ----------------
def make_quant(Xtr, bits=BITS, q=(0.5, 99.5)):
    """True k-bit quantization: clip to the training [q_lo, q_hi] range, then 2^k levels per feature."""
    lo = np.percentile(Xtr, q[0], axis=0); hi = np.percentile(Xtr, q[1], axis=0)
    span = np.maximum(hi - lo, 1e-8); L = 2 ** bits - 1
    def sq(X):
        Z = np.clip((X - lo) / span, 0, 1)
        return (np.round(Z * L) / L * span + lo).astype(np.float32)
    return sq

def paper_sq(X):
    """The original paper code: round(x*256)/256, then a trailing median over 3 adjacent feature columns."""
    X1 = np.round(X * 256) / 256
    s1 = np.full_like(X1, np.nan); s1[:, 1:] = X1[:, :-1]
    s2 = np.full_like(X1, np.nan); s2[:, 2:] = X1[:, :-2]
    return np.nanmedian(np.stack([X1, s1, s2]), axis=0).astype(np.float32)

# ---------------- SCORES ----------------
def conf(p): return np.maximum(p, 1 - p)

def base_probs(X, rf, xgb):
    return rf.predict_proba(X)[:, 1], xgb.predict_proba(X)[:, 1]

def independent_scores(X, rf, xgb, rng):
    """Detectors that do not depend on the squeezer."""
    p_rf, p_xg = base_probs(X, rf, xgb)
    pn = rf.predict_proba(X + rng.normal(0, 0.01, X.shape).astype(np.float32))[:, 1]
    return {"Noise sensitivity (RF)": np.abs(p_rf - pn),
            "Ensemble disagreement only": np.abs(p_rf - p_xg)}

def squeeze_scores(X, rf, xgb, sq):
    p_rf, p_xg = base_probs(X, rf, xgb)
    Xs = sq(X)
    s_rf, s_xg = base_probs(Xs, rf, xgb)
    return {"Feature squeezing (RF)": np.abs(p_rf - s_rf),
            "Confidence drop (RF)": np.abs(conf(p_rf) - conf(s_rf)),
            "Squeeze + ensemble, Eq.11 variance": np.var(np.vstack([s_rf, s_xg]), axis=0),
            "Squeeze + ensemble, Eq.24 sum": np.abs(p_rf - s_rf) + np.abs(p_xg - s_xg)}

PROPOSED = ["Squeeze + ensemble, Eq.11 variance", "Squeeze + ensemble, Eq.24 sum"]
BASELINES = ["Noise sensitivity (RF)", "Feature squeezing (RF)", "Confidence drop (RF)"]

# ---------------- METRICS ----------------
def metrics(neg_val, neg_te, pos_te):
    thr = np.percentile(neg_val, FAR_TARGET_PCT)
    tp = int((pos_te > thr).sum()); fn = len(pos_te) - tp
    fp = int((neg_te > thr).sum()); tn = len(neg_te) - fp
    tpr, fpr = tp / len(pos_te), fp / len(neg_te)
    prec = tpr / (tpr + fpr + 1e-12)                  # precision at balanced prevalence
    f1 = 2 * prec * tpr / (prec + tpr + 1e-12)
    y = np.r_[np.zeros(len(neg_te)), np.ones(len(pos_te))]; s = np.r_[neg_te, pos_te]
    return dict(thr=thr, TPR=tpr, FPR=fpr, Precision_bal=prec, F1_bal=f1,
                AUROC=roc_auc_score(y, s), AUPRC=average_precision_score(y, s),
                TP=tp, FN=fn, FP=fp, TN=tn)


In [ ]:
# ---- AUTOENCODER BASELINE (reconstruction error): same splits, threshold and metrics as every other detector ----
from sklearn.neural_network import MLPRegressor
AE_HIDDEN = (64, 16, 64)          # encoder 64 -> bottleneck 16 -> decoder 64
AE_TRAIN_CAP = 30000              # clean training flows used for the autoencoder

def split_sets(D, seed):          # identical sampling to the extension run (all features)
    role = split_roles(seed); rng = np.random.RandomState(seed)
    g = lambda k, sp, cap: take(D[k], role, sp, rng, cap)
    Xb, Xo = g("benign", "train", 10**9), g("orig", "train", 10**9)
    n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
    Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
    return dict(train=(np.vstack([Xb, Xo]), np.r_[np.zeros(n), np.ones(n)]),
                val=np.vstack([g("benign", "val", MAX_EVAL_PER_SET), g("orig", "val", MAX_EVAL_PER_SET)]),
                neg=np.vstack([g("benign", "test", MAX_EVAL_PER_SET), g("orig", "test", MAX_EVAL_PER_SET)]),
                adv=np.vstack([g(k, "test", MAX_EVAL_PER_SET) for k in ATTACKS]))

rows = []
for ds in FAMILIES:
    print("loading", ds)
    D, feats = load_dataset(ds)
    for seed in SEEDS:
        t0 = time.time()
        S = split_sets(D, seed); Xtr, ytr = S["train"]
        sc = StandardScaler().fit(Xtr); Z = sc.transform(Xtr).astype(np.float32)
        T = lambda X, sc=sc: sc.transform(X).astype(np.float32)
        val, neg, adv = T(S["val"]), T(S["neg"]), T(S["adv"])
        rf = RandomForestClassifier(150, max_depth=20, n_jobs=-1, random_state=seed).fit(Z, ytr)
        xg = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss", random_state=seed).fit(Z, ytr)
        idx = np.random.RandomState(seed).choice(len(Z), min(len(Z), AE_TRAIN_CAP), replace=False)
        ae = MLPRegressor(hidden_layer_sizes=AE_HIDDEN, activation="relu", solver="adam", batch_size=256, max_iter=60,
                          early_stopping=True, n_iter_no_change=5, random_state=seed).fit(Z[idx], Z[idx])
        dets = {"Autoencoder (reconstruction error)": lambda X: ((ae.predict(X) - X) ** 2).mean(axis=1),
                "Ensemble disagreement only": lambda X: np.abs(rf.predict_proba(X)[:, 1] - xg.predict_proba(X)[:, 1])}
        for name, f in dets.items():
            m = metrics(f(val), f(neg), f(adv))
            rows.append(dict(dataset=ds, seed=seed, method=name, TPR=m["TPR"], FPR=m["FPR"], AUROC=m["AUROC"], AUPRC=m["AUPRC"],
                             TP=m["TP"], FN=m["FN"], FP=m["FP"], TN=m["TN"], ae_epochs=ae.n_iter_ if "Auto" in name else np.nan))
        print(f"  {ds} seed {seed} done in {time.time() - t0:.0f}s (autoencoder trained {ae.n_iter_} epochs)")
    del D

ae_res = pd.DataFrame(rows); ae_res.to_csv(f"{OUT}/autoencoder_results.csv", index=False)
pm = lambda x: f"{np.mean(x):.3f} +/- {np.std(x, ddof=1) if len(x) > 1 else 0:.3f}"
summ = ae_res.groupby(["dataset", "method"]).agg(TPR=("TPR", pm), FPR=("FPR", pm), AUROC=("AUROC", pm), AUPRC=("AUPRC", pm)).reset_index()
summ.to_csv(f"{OUT}/autoencoder_summary.csv", index=False)
pd.set_option("display.width", 220)
print("\n===== AUTOENCODER vs ENSEMBLE DISAGREEMENT (4 attacks pooled, threshold = 95th percentile of clean validation scores) =====")
print(summ.to_string(index=False))
print("\n===== Paired t-test on AUROC: ensemble disagreement vs autoencoder =====")
for ds in FAMILIES:
    a = ae_res[(ae_res.dataset == ds) & (ae_res.method == "Ensemble disagreement only")].sort_values("seed").AUROC.values
    b = ae_res[(ae_res.dataset == ds) & (ae_res.method == "Autoencoder (reconstruction error)")].sort_values("seed").AUROC.values
    t, p = stats.ttest_rel(a, b)
    print(f"{ds:10s} ensemble {a.mean():.3f} vs autoencoder {b.mean():.3f}  diff={a.mean() - b.mean():+.3f}  t={t:.2f}  p={p:.4f}  ensemble wins {int((a > b).sum())}/{len(a)}")
print("(Seeds share the same underlying flows, so p-values are indicative, not formal.)")
